# AI-Powered Customer Churn Intelligence System

## Day 2 - Data Cleaning & Feature Engineering

This notebook prepares the customer churn dataset for SQL analysis and machine learning.

The workflow includes:

- Data loading
- Data type validation
- Duplicate detection
- Categorical value validation
- Date conversion
- Target encoding
- Feature engineering
- Final dataset validation
- Export of the processed dataset

In [1]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Load raw dataset
df = pd.read_csv("../data/customer_subscription_churn_usage_patterns.csv")

print("Raw dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Raw dataset loaded successfully.
Rows: 2,800
Columns: 10


## 1. Initial Data Validation

The raw dataset is inspected before applying any transformations. This ensures that cleaning operations are based on the actual structure and quality of the source data.

In [2]:
# Dataset structure
print("Data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Data types:
user_id                     int64
signup_date                object
plan_type                  object
monthly_fee                 int64
avg_weekly_usage_hours    float64
support_tickets             int64
payment_failures            int64
tenure_months               int64
last_login_days_ago         int64
churn                      object
dtype: object

Missing values:
user_id                   0
signup_date               0
plan_type                 0
monthly_fee               0
avg_weekly_usage_hours    0
support_tickets           0
payment_failures          0
tenure_months             0
last_login_days_ago       0
churn                     0
dtype: int64

Duplicate rows:
0


In [3]:
# Validate categorical columns

print("Plan types:")
print(df["plan_type"].unique())

print("\nChurn values:")
print(df["churn"].unique())

Plan types:
['Premium' 'Basic' 'Standard']

Churn values:
['Yes' 'No']


In [4]:
# Check customer ID uniqueness

print("Total customer records:", len(df))
print("Unique customer IDs:", df["user_id"].nunique())

if df["user_id"].nunique() == len(df):
    print("Each customer ID is unique.")
else:
    print("Duplicate customer IDs detected.")

Total customer records: 2800
Unique customer IDs: 2800
Each customer ID is unique.


In [5]:
# Numerical range validation

numerical_columns = [
    "monthly_fee",
    "avg_weekly_usage_hours",
    "support_tickets",
    "payment_failures",
    "tenure_months",
    "last_login_days_ago"
]

df[numerical_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
monthly_fee,2800.0,434.214286,205.678472,199.0,199.0,399.0,699.0,699.0
avg_weekly_usage_hours,2800.0,12.891429,7.109691,0.5,6.7,12.8,19.2,25.0
support_tickets,2800.0,3.887857,2.606419,0.0,2.0,4.0,6.0,8.0
payment_failures,2800.0,2.491786,1.691647,0.0,1.0,2.0,4.0,5.0
tenure_months,2800.0,18.612857,10.374487,1.0,10.0,18.0,27.0,36.0
last_login_days_ago,2800.0,30.005000,17.852757,0.0,14.0,30.0,46.0,60.0


In [6]:
df.dtypes

user_id                     int64
signup_date                object
plan_type                  object
monthly_fee                 int64
avg_weekly_usage_hours    float64
support_tickets             int64
payment_failures            int64
tenure_months               int64
last_login_days_ago         int64
churn                      object
dtype: object

In [7]:
df["plan_type"].unique()
df["churn"].unique()

array(['Yes', 'No'], dtype=object)

## 2. Data Type Cleaning

The `signup_date` column is converted from a text representation into a proper datetime format so that time-based features can be created reliably.

In [8]:
# Convert signup_date to datetime

df["signup_date"] = pd.to_datetime(
    df["signup_date"],
    errors="coerce"
)

print("signup_date data type:", df["signup_date"].dtype)
print("Invalid dates:", df["signup_date"].isnull().sum())

signup_date data type: datetime64[ns]
Invalid dates: 0


## 3. Duplicate Record Handling

Duplicate rows are checked and removed to ensure that each customer record is represented only once in the analytical dataset.

In [9]:
# Remove duplicate rows

initial_rows = len(df)

df = df.drop_duplicates().copy()

removed_duplicates = initial_rows - len(df)

print("Initial rows:", initial_rows)
print("Rows after duplicate removal:", len(df))
print("Duplicates removed:", removed_duplicates)

Initial rows: 2800
Rows after duplicate removal: 2800
Duplicates removed: 0


## 4. Target Variable Encoding

The `churn` column contains categorical values (`Yes` and `No`). A numerical target variable is created for machine learning while retaining the original churn column for business analysis.

In [10]:
# Encode churn target

df["churn_target"] = df["churn"].map({
    "No": 0,
    "Yes": 1
})

print(df[["churn", "churn_target"]].drop_duplicates())

  churn  churn_target
0   Yes             1
9    No             0


In [11]:
# Validate target encoding

print("Missing target values:", df["churn_target"].isnull().sum())
print("\nTarget distribution:")
print(df["churn_target"].value_counts())

Missing target values: 0

Target distribution:
churn_target
1    1605
0    1195
Name: count, dtype: int64


In [12]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 2800
Columns: 11


## 5. Customer Engagement Features

Customer engagement is an important factor in churn analysis. The number of days since a customer's last login is transformed into an interpretable engagement category.

The categories are:

- Active: 0–14 days since last login
- At Risk: 15–30 days
- Inactive: More than 30 days

In [13]:
# Create customer engagement category

df["login_recency_category"] = pd.cut(
    df["last_login_days_ago"],
    bins=[-1, 14, 30, float("inf")],
    labels=["Active", "At Risk", "Inactive"]
)

print(df["login_recency_category"].value_counts())

login_recency_category
Inactive    1350
At Risk      741
Active       709
Name: count, dtype: int64


## 6. Support Risk Feature

Support-ticket volume is transformed into a simple support-risk category to help identify customers experiencing higher levels of support interaction.

In [14]:
# Create support risk category

df["support_risk"] = pd.cut(
    df["support_tickets"],
    bins=[-1, 2, 5, float("inf")],
    labels=["Low", "Medium", "High"]
)

print(df["support_risk"].value_counts())

support_risk
Low       994
High      924
Medium    882
Name: count, dtype: int64


## 7. Payment Risk Feature

Payment failures are converted into a categorical payment-risk feature to support customer segmentation and later business analysis.

In [15]:
# Create payment risk category

df["payment_risk"] = pd.cut(
    df["payment_failures"],
    bins=[-1, 0, 2, float("inf")],
    labels=["Low", "Medium", "High"]
)

print(df["payment_risk"].value_counts())

payment_risk
High      1384
Medium     974
Low        442
Name: count, dtype: int64


## 8. Usage Level Feature

Weekly usage hours are categorized into usage levels to make customer engagement patterns easier to interpret during business analysis.

In [16]:
# Create usage level category

df["usage_level"] = pd.cut(
    df["avg_weekly_usage_hours"],
    bins=[-1, 5, 15, float("inf")],
    labels=["Low", "Medium", "High"]
)

print(df["usage_level"].value_counts())

usage_level
High      1177
Medium    1100
Low        523
Name: count, dtype: int64


In [17]:
# Preview engineered features

engineered_features = [
    "last_login_days_ago",
    "login_recency_category",
    "support_tickets",
    "support_risk",
    "payment_failures",
    "payment_risk",
    "avg_weekly_usage_hours",
    "usage_level",
    "churn_target"
]

df[engineered_features].head(10)

,last_login_days_ago,login_recency_category,support_tickets,support_risk,payment_failures,payment_risk,avg_weekly_usage_hours,usage_level,churn_target
0,14,Active,4,Medium,1,Medium,1.1,Low,1
1,1,Active,6,High,0,Low,2.6,Low,1
2,14,Active,8,High,3,High,14.3,Medium,1
3,9,Active,5,Medium,2,Medium,17.6,High,1
4,38,Inactive,5,Medium,2,Medium,9.8,Medium,1
5,35,Inactive,6,High,0,Low,13.6,Medium,1
6,42,Inactive,1,Low,0,Low,14.6,Medium,1
7,29,At Risk,6,High,2,Medium,21.7,High,1
8,59,Inactive,4,Medium,5,High,9.2,Medium,1
9,29,At Risk,3,Medium,1,Medium,13.6,Medium,0


In [18]:
print("Dataset shape after feature engineering:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset shape after feature engineering:
Rows: 2800
Columns: 15


In [19]:
df.shape

(2800, 15)

In [20]:
df[[
    "login_recency_category",
    "support_risk",
    "payment_risk",
    "usage_level"
]].head()

,login_recency_category,support_risk,payment_risk,usage_level
0,Active,Medium,Medium,Low
1,Active,High,Low,Low
2,Active,High,High,Medium
3,Active,Medium,Medium,High
4,Inactive,Medium,Medium,Medium


## 9. Export Processed Dataset

The cleaned and feature-engineered dataset is exported separately from the original raw dataset.

The original source data is preserved unchanged so that the analytical workflow remains reproducible.

In [21]:
# Final validation

print("========== FINAL DATA VALIDATION ==========")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

========== FINAL DATA VALIDATION ==========
Rows: 2,800
Columns: 15

Missing values:
user_id                   0
signup_date               0
plan_type                 0
monthly_fee               0
avg_weekly_usage_hours    0
support_tickets           0
payment_failures          0
tenure_months             0
last_login_days_ago       0
churn                     0
churn_target              0
login_recency_category    0
support_risk              0
payment_risk              0
usage_level               0
dtype: int64

Duplicate rows:
0

Data types:
user_id                            int64
signup_date               datetime64[ns]
plan_type                         object
monthly_fee                        int64
avg_weekly_usage_hours           float64
support_tickets                    int64
payment_failures                   int64
tenure_months                      int64
last_login_days_ago                int64
churn                             object
churn_target                       int64

In [22]:
# Export processed dataset

output_path = "../data/customer_churn_processed.csv"

df.to_csv(output_path, index=False)

print(f"Processed dataset saved to: {output_path}")

Processed dataset saved to: ../data/customer_churn_processed.csv


In [23]:
# Verify exported dataset

processed_df = pd.read_csv(output_path)

print("Processed dataset successfully verified.")
print("Shape:", processed_df.shape)
print("\nColumns:")
print(processed_df.columns.tolist())

Processed dataset successfully verified.
Shape: (2800, 15)

Columns:
['user_id', 'signup_date', 'plan_type', 'monthly_fee', 'avg_weekly_usage_hours', 'support_tickets', 'payment_failures', 'tenure_months', 'last_login_days_ago', 'churn', 'churn_target', 'login_recency_category', 'support_risk', 'payment_risk', 'usage_level']
